In [2]:
# Install TensorFlow (already installed usually)
!pip install tensorflow

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ---------------------------
# Dataset (Directly inside code)
# ---------------------------
text = """once upon a time there was a king
the king had a queen
the queen loved the king
the kingdom was peaceful"""

text = text.lower()

# ---------------------------
# Tokenization
# ---------------------------
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

# ---------------------------
# Create Sequences
# ---------------------------
input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# ---------------------------
# Model
# ---------------------------
model = Sequential([
    Embedding(total_words, 50, input_length=max_len-1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')

# ---------------------------
# Train
# ---------------------------
model.fit(X, y, epochs=10, verbose=1)

# ---------------------------
# Generate Text
# ---------------------------
def generate_text(seed, n_words):
    for _ in range(n_words):
        seq = tokenizer.texts_to_sequences([seed])[0]
        seq = pad_sequences([seq], maxlen=max_len-1, padding='pre')
        pred = np.argmax(model.predict(seq), axis=-1)[0]

        for word, index in tokenizer.word_index.items():
            if index == pred:
                seed += " " + word
                break
    return seed

print(generate_text("once upon", 5))
print(generate_text("the king", 5))

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - loss: 2.6411
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 2.6355
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 2.6298
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 2.6240
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 2.6180
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 2.6116
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 2.6047
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 2.5972
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 2.5889
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 2.5796
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
once upon king king king king king
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━